# Заняття 6 · Демо 1 · SAM і SAM 2.1: промпт, неоднозначність і де зникає час

**LIVE DEMO 1 (слайд 25)** · слайди 11–24

SAM — вузол «обвести» в Grounded-SAM: дає піксельну маску за точкою чи боксом, але не знає, *що* саме обвів.
Розбираємо, з чого він складається, де він ламається на аерокадрі й скільки коштує кожен вузол.
Основна модель — **SAM 2.1 Hiera-L** (image API офіційного пакета `sam2`, той самий, що на слайді 22); для порівняння
поколінь — SAM ViT-B / ViT-H, HQ-SAM, MobileSAM і FastSAM. Дані — **VisDrone2019-DET-val**: 548 кадрів, 38 759 об'єктів.

| Розділ | Що показуємо | Слайди |
|---|---|---|
| 1. Скільки пікселів лишається від цілі | Розміри всіх об'єктів VisDrone-val до і після ресайзу в 1024; ціль на сітці патчів 16×16 | 13, 14 |
| 2. Важкий енкодер, легкий декодер | Параметри по вузлах, embedding 256×64×64, що в ньому лежить | 12, 13, 15 |
| 3. Клік → три маски | `multimask_output`, `predicted_iou`, інтерактивний клік мишею | 16, 17, 21 |
| 4. Промпти й уточнення | Точка фону, бокс, `mask_input` з попереднього кроку, батч боксів | 15, 21, 22 |
| 5. `predicted_iou` ≠ впевненість | Клік по фону теж дає «якісну» маску; дрібні цілі вироджуються; кроп рятує | 14, 17, 18 |
| 6. Автоматичний режим | Сітка точок, `crop_n_layers`, FastSAM: покриття, сміття, час | 18, 21, 23 |
| 7. Де зникає час | Енкодер vs декодер по сімейству, T(k) = T_enc + k·T_dec, CPU | 19, 24 |

**Підготовка (один раз, з мережею):**
```bash
# з кореня репозиторію
python tools/fetch_visdrone.py --dataset visdrone-det   # VisDrone2019-DET-val, ~80 МБ -> artifacts/visdrone/
python tools/fetch_sam_models.py                        # SAM, HQ-SAM, SAM 2.1 T/S/B+/L, MobileSAM, FastSAM -> artifacts/models/
```
Повний прогін — 2–3 хвилини на GPU з 12+ ГБ. На 8 ГБ приберіть `sam-vit-huge` і `sam2.1-large` з `BENCH`; на CPU ноутбук
сам візьме `sam2.1-tiny` і скоротить бенчмарк.

In [ ]:
from pathlib import Path

# ---------------- Конфігурація ----------------
DEVICE = None                  # None — cuda / mps / cpu автоматично
SAM_MAIN = "sam2.1-large"      # модель для розділів 2–6
BENCH = ["sam-vit-base", "sam-vit-huge", "sam-hq-vit-base", "mobile-sam",
         "sam2.1-tiny", "sam2.1-small", "sam2.1-base-plus", "sam2.1-large"]   # розділ 7
SAM_INPUT, PATCH = 1024, 16    # фіксований вхід енкодера SAM / SAM 2 і розмір патча ViT
N_FRAMES_STATS = 30            # кадрів для розділу 5
SEED = 0
OUT = Path("outputs/demo1")    # картинки й таблиці для слайдів

import panels as P
hf = P.setup_hf(sam=[SAM_MAIN, *BENCH, "fastsam"])   # офлайн, якщо всі ваги вже в маніфесті

import numpy as np, pandas as pd, torch, cv2
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
DEVICE = P.pick_device(DEVICE)
if DEVICE == "cpu":            # енкодер на CPU — ~0.5 с на кадр: легша модель і менша вибірка розділу 5
    SAM_MAIN, BENCH = "sam2.1-tiny", ["sam-vit-base", "mobile-sam", "sam2.1-tiny"]
    N_FRAMES_STATS = 8
GPU = P.gpu_name(DEVICE)
rng = np.random.default_rng(SEED)
OUT.mkdir(parents=True, exist_ok=True)

items = P.det_items()
print(f"HF: {hf} | {DEVICE}: {GPU} | VisDrone-DET-val: {len(items)} кадрів, {sum(len(g.boxes) for _, g in items)} об'єктів")

## 1. Скільки пікселів лишається від цілі

Енкодер SAM приймає **фіксований вхід 1024 × 1024**: довша сторона кадру стискається до 1024, далі ViT ріже кадр на патчі
16 × 16 (слайд 13). Рахуємо для кожного з ~39 тис. об'єктів VisDrone-val максимальну сторону на сирому кадрі й після цього
ресайзу. Ціль, менша за 16 px на вході, займає **менше одного токена** — про її форму енкодер майже нічого не знає (слайд 14).

In [ ]:
tokens = (SAM_INPUT // PATCH) ** 2
print(f"{SAM_INPUT} / {PATCH} = {SAM_INPUT // PATCH} → {SAM_INPUT // PATCH}² = {tokens} токенів; "
      f"embedding: 256 × {SAM_INPUT // PATCH} × {SAM_INPUT // PATCH}")

raw, scaled, frame_w = [], [], []
for p, gt in items:
    w, h = P.image_size(p)
    side = np.maximum(gt.boxes[:, 2] - gt.boxes[:, 0], gt.boxes[:, 3] - gt.boxes[:, 1])
    raw.append(side)
    scaled.append(side * SAM_INPUT / max(w, h))     # ← ресайз довшої сторони в 1024
    frame_w.append(np.full(len(side), max(w, h)))
raw, scaled, frame_w = map(np.concatenate, (raw, scaled, frame_w))

edges = [0, 8, 16, 32, 64, 128, np.inf]
labels = ["< 8", "8–16", "16–32", "32–64", "64–128", "≥ 128"]
dist = pd.DataFrame({"сирий кадр, %": np.histogram(raw, edges)[0] / len(raw) * 100,
                     f"вхід SAM {SAM_INPUT}, %": np.histogram(scaled, edges)[0] / len(raw) * 100}, index=labels)
print(f"довша сторона кадру: медіана {np.median(frame_w):.0f} px → коефіцієнт ресайзу ×{SAM_INPUT / np.median(frame_w):.2f}")
print(f"цілей < {PATCH} px: сирий кадр {np.mean(raw < PATCH):.1%} → вхід SAM {np.mean(scaled < PATCH):.1%}")

fig, ax = plt.subplots(figsize=(8, 3.4))
dist.plot.bar(ax=ax, color=["#5b6676", "#2bb5a0"], rot=0, width=0.8)
for c in ax.containers:
    ax.bar_label(c, fmt="%.1f", fontsize=8)
ax.set_xlabel("максимальна сторона об'єкта, px"); ax.set_ylabel("% об'єктів")
ax.set_title(f"VisDrone-val, {len(raw)} об'єктів: до і після ресайзу в енкодер SAM")
P.save_fig(fig, OUT / "01_object_size_hist.png")

Як це виглядає: та сама ціль на сирому кадрі й на вході енкодера, поверх — сітка патчів 16 × 16, з якої ViT робить токени.

In [ ]:
# об'єкт ~27 px на сирому кадрі 1920 — як на слайді 14
cand = [(p, b) for p, gt in items[:200] for b in gt.boxes
        if 25 <= max(b[2] - b[0], b[3] - b[1]) <= 29 and P.image_size(p)[0] >= 1900]
p, b = cand[len(cand) // 2]
img = P.imread_rgb(p)
s = SAM_INPUT / max(img.shape[:2])
small = cv2.resize(img, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)       # що бачить енкодер


def zoom(im, box, cell, pad, out=360):
    # кроп навколо цілі без згладжування + сітка токенів ViT (cell — розмір токена в пікселях цього кадру)
    c, (x0, y0) = P.crop_around(im, box, pad)
    vis = P.draw_boxes(c, [np.asarray(box) - [x0, y0, x0, y0]], thick=1)
    z = out / max(c.shape[:2])
    vis = cv2.resize(vis, None, fx=z, fy=z, interpolation=cv2.INTER_NEAREST)
    for gx in np.arange((-x0) % cell, c.shape[1], cell):
        vis[:, min(int(gx * z), vis.shape[1] - 1)] = 255
    for gy in np.arange((-y0) % cell, c.shape[0], cell):
        vis[min(int(gy * z), vis.shape[0] - 1)] = 255
    return vis


side_raw = max(b[2] - b[0], b[3] - b[1])
P.show_row([zoom(img, b, PATCH / s, pad=40), zoom(small, b * s, PATCH, pad=40 * s)],
           [f"сирий кадр {img.shape[1]} px: ціль {side_raw:.0f} px",
            f"вхід SAM {SAM_INPUT} px: ціль {side_raw * s:.0f} px ≈ {(side_raw * s / PATCH) ** 2:.1f} токена"],
           size=4.2, suptitle=f"Сітка — токени ViT (16 px входу = {PATCH / s:.0f} px сирого кадру). Ціль ледь на один токен")

**Підсумок розділу.** Понад чверть цілей VisDrone-val на вході SAM займають менше одного патч-токена, хоча на сирому кадрі
таких лише ~16 %: ресайз довшої сторони до 1024 стискає типовий кадр 1360 px у ×0.75, а 1920 px — майже вдвічі (×0.53)
ще до першого шару (960-піксельні кадри, навпаки, збільшуються). Для 4K-кадру коефіцієнт
×0.27 — ще гірше. Розв'язок не у важчій моделі, а в тайлах / кропах (розділ 5, слайд 46).

## 2. Важкий енкодер, легкий декодер

SAM — це один важкий **image encoder** на кадр і легкі **prompt encoder + mask decoder** на кожен промпт (слайд 12).
У SAM 2 до них додаються вузли пам'яті для відео (слайд 28) — для зображень вони просто не працюють. Рахуємо параметри
по вузлах і дивимось, що лежить в embedding, який кешує `set_image()`.

In [ ]:
from sam2.sam2_image_predictor import SAM2ImagePredictor

pred = P.load_sam2_image(SAM_MAIN, DEVICE)        # = SAM2ImagePredictor(build_sam2(конфіг, чекпойнт))
m = pred.model
nodes = {"image_encoder (Hiera)": m.image_encoder, "sam_prompt_encoder": m.sam_prompt_encoder,
         "sam_mask_decoder": m.sam_mask_decoder, "memory_attention (відео)": m.memory_attention,
         "memory_encoder (відео)": m.memory_encoder}
total = P.n_params(m)
display(pd.DataFrame({"параметрів, млн": [P.n_params(v) / 1e6 for v in nodes.values()],
                      "% моделі": [P.n_params(v) / total * 100 for v in nodes.values()]}, index=list(nodes)).round(2))

# ціль для розділів 2–6: кадр із багатьма машинами середнього розміру, найбільша машина в ньому
def score_item(it):
    p, gt = it
    side = np.maximum(gt.boxes[:, 2] - gt.boxes[:, 0], gt.boxes[:, 3] - gt.boxes[:, 1])
    return (np.isin(gt.cats, [4, 5]) & (side > 25) & (side < 90)).sum()

IMG_PATH, GT0 = max(items, key=score_item)
img = P.imread_rgb(IMG_PATH)
H, W = img.shape[:2]
cars = GT0.boxes[np.isin(GT0.cats, [4, 5])]
tgt = cars[np.argmax((cars[:, 2] - cars[:, 0]) * (cars[:, 3] - cars[:, 1]))]
cx, cy = (tgt[0] + tgt[2]) / 2, (tgt[1] + tgt[3]) / 2

t_enc = []
with torch.inference_mode(), P.autocast(DEVICE):
    pred.set_image(img)                                # прогрів
    with P.timed(t_enc):
        pred.set_image(img)                            # ← енкодер: раз на кадр, результат кешується
emb = pred._features["image_embed"][0].float().cpu().numpy()
print(f"{IMG_PATH.name}: {W}×{H}, машин {len(cars)} | set_image: {t_enc[0]:.0f} мс | image_embed {tuple(emb.shape)} "
      f"+ high-res ознаки {[tuple(f.shape[1:]) for f in pred._features['high_res_feats']]}")

In [ ]:
# Що в embedding: 3 головні компоненти PCA по 256 каналах → RGB. Об'єкти одного типу мають схожий колір —
# семантика в ознаках є, але назви в SAM немає: декодер її не використовує (слайд 9).
X = emb.reshape(256, -1).T
X = X - X.mean(0)
_, _, Vt = np.linalg.svd(X, full_matrices=False)
pc = X @ Vt[:3].T
pc = (pc - np.percentile(pc, 1, 0)) / (np.percentile(pc, 99, 0) - np.percentile(pc, 1, 0))
g = SAM_INPUT // PATCH
pca = (np.clip(pc, 0, 1).reshape(g, g, 3) * 255).astype(np.uint8)
s = SAM_INPUT / max(H, W)
pca = pca[:int(np.ceil(H * s / PATCH)), :int(np.ceil(W * s / PATCH))]      # прибираємо паддінг до квадрата
P.show_row([img, cv2.resize(pca, (W, H), interpolation=cv2.INTER_NEAREST)],
           ["кадр", f"image_embed 256×{g}×{g}: 3 компоненти PCA (одна клітинка = один токен)"], size=6)

**Підсумок розділу.** Понад 90 % параметрів — в енкодері; prompt encoder — тисячі параметрів, декодер — кілька мільйонів.
Embedding 256 × 64 × 64 рахується раз на кадр і перевикористовується для будь-якої кількості промптів — звідси
«кешуйте embedding» (слайд 12). Сітка 64 × 64 в embedding — та сама межа роздільності, що й у розділі 1.

## 3. Клік → три маски

Клік у машину неоднозначний: дах, машина чи вся група машин? З `multimask_output=True` SAM повертає **три маски**
(subpart / part / whole) і `predicted_iou` для кожної (слайди 16–17). Модель не вибирає за вас — вибирає ваш код.

In [ ]:
with torch.inference_mode(), P.autocast(DEVICE):
    masks, iou, low = pred.predict(point_coords=np.array([[cx, cy]]), point_labels=np.array([1]),
                                   multimask_output=True)
masks = masks > 0
order = np.argsort([mk.sum() for mk in masks])                 # від найменшої до найбільшої
crop, (x0, y0) = P.crop_around(img, tgt, pad=90)
panels = []
for k in order:
    v = P.overlay_masks(crop, [masks[k][y0:y0 + crop.shape[0], x0:x0 + crop.shape[1]]], alpha=.55)
    cv2.circle(v, (int(cx - x0), int(cy - y0)), 4, (255, 60, 60), -1)
    panels.append(v)
P.show_row(panels, [f"{n} · predicted_iou {iou[k]:.2f} · {int(masks[k].sum())} px"
                    for n, k in zip(["subpart", "part", "whole"], order)],
           size=4, suptitle="Один клік — три валідні відповіді (впорядковано за площею)", save=OUT / "01_ambiguity_3masks.png")

**Спробуйте самі.** Клік — точка об'єкта, Shift+клік (або перемикач) — точка фону. Енкодер уже порахований, тож кожен клік —
лише декодер: мілісекунди (заголовок під картинкою). Зніміть `multimask_output` після кількох точок — модель упевнена, що
ви хочете одну маску. Спробуйте клікнути в дрібну машину далеко від камери і в тінь від будинку.

In [ ]:
def predict_fn(points, labels, multimask):
    ts = []
    with torch.inference_mode(), P.autocast(DEVICE), P.timed(ts):
        m_, iou_, _ = pred.predict(point_coords=points, point_labels=labels, multimask_output=multimask)
    return m_ > 0, iou_, ts[0]

view = np.array([max(0, cx - 480), max(0, cy - 270), min(W, cx + 480), min(H, cy + 270)], int)
panel = P.click_sam_panel(img, predict_fn, view_box=view, start_point=(cx, cy))

**Підсумок розділу.** Неоднозначність — властивість запиту, а не помилка моделі: архітектура з однією маскою на двозначний
клік навчилася б усередненого компромісу. Три виходи + `predicted_iou` перекладають вибір на ваш код: найбільший
`predicted_iou`, найбільша площа або маска, що найкраще лягає в бокс детектора.

## 4. Промпти й уточнення

* **Точка фону** (`label=0`) вирізає зайве.
* **Бокс** — дві спеціальні точки-кути; з ним `multimask_output=False`, бо неоднозначності майже немає (слайд 17).
* **`mask_input`** — low-res логіти 256 × 256 з попереднього кроку: ітеративне уточнення, на якому тримаються корекція й відео (слайд 15).
* **Батч боксів** — режим «промпт із моделі» (слайд 21): усі бокси детектора одним викликом декодера.

In [ ]:
best = int(np.argmax(iou))
bx, by = float(min(tgt[2] + 12, W - 1)), float(cy)          # точка фону: поруч із ціллю, поза її боксом
with torch.inference_mode(), P.autocast(DEVICE):
    m_neg, iou_neg, _ = pred.predict(point_coords=np.array([[cx, cy], [bx, by]]), point_labels=np.array([1, 0]),
                                     multimask_output=False)
    m_ref, iou_ref, _ = pred.predict(point_coords=np.array([[cx, cy], [bx, by]]), point_labels=np.array([1, 0]),
                                     mask_input=low[best][None], multimask_output=False)
    m_box, iou_box, _ = pred.predict(box=tgt, multimask_output=False)

crop_panels = []
for t, mk, s_ in [("клік, найкраща з 3", masks[best], iou[best]), ("клік + точка фону", m_neg[0] > 0, iou_neg[0]),
                  ("+ mask_input з кроку 1", m_ref[0] > 0, iou_ref[0]), ("бокс", m_box[0] > 0, iou_box[0])]:
    v = P.overlay_masks(crop, [mk[y0:y0 + crop.shape[0], x0:x0 + crop.shape[1]]], alpha=.55)
    crop_panels.append((v, f"{t}\npredicted_iou {s_:.2f}"))
P.show_row([v for v, _ in crop_panels], [t for _, t in crop_panels], size=3.6, save=OUT / "01_prompt_types.png")

# режим 03 «промпт із моделі»: усі GT-бокси машин як «ідеальний детектор», одним батчем
ts = []
with torch.inference_mode(), P.autocast(DEVICE), P.timed(ts):
    mb, iou_b, _ = pred.predict(box=cars, multimask_output=False)
mb = mb.reshape(len(cars), H, W) > 0
print(f"батч із {len(cars)} боксів: {ts[0]:.1f} мс на всі ({ts[0] / len(cars):.2f} мс на маску) — енкодер не перераховується")
P.show_row([P.overlay_masks(img, mb, alpha=.5, contour=False)], [f"{len(cars)} боксів → {len(cars)} масок за один виклик"], size=11)

**Підсумок розділу.** Будь-який промпт — це кілька токенів для декодера: точку, бокс чи маску з попереднього кроку може
згенерувати людина, детектор або трекер. Саме тому SAM — будівельний блок (слайд 15), а Grounded-SAM — просто детектор,
що пише бокси замість людини (демо 3).

## 5. `predicted_iou` ≠ впевненість, а дрібні цілі вироджуються

Два експерименти на `N_FRAMES_STATS` кадрах VisDrone-val (слайди 14, 17, 18):
1. **Клік по фону.** Ставимо точки в місця, де немає жодного GT-об'єкта (дорога, дах, газон), і порівнюємо `predicted_iou`
   з кліками в центри машин. Якщо розподіли перекриваються — поріг на `predicted_iou` не відрізнить машину від шматка асфальту.
2. **Розмір цілі.** Для кліку в центр GT-машини міряємо, наскільки бокс маски збігається з GT-боксом (IoU), окремо за
   розміром цілі *на вході SAM*. Потім той самий клік, але SAM бачить **кроп навколо цілі** (4 її сторони, не менше 256 px,
   тобто збільшення в 4–7 разів перед енкодером) — це і є тайлінг / SAHI для SAM.

In [ ]:
from tqdm.auto import tqdm

stat_items = [items[i] for i in rng.choice(len(items), N_FRAMES_STATS, replace=False)]
rows = []
for p, gt in tqdm(stat_items, desc="кадри"):
    im = P.imread_rgb(p)
    h, w = im.shape[:2]
    s = SAM_INPUT / max(h, w)
    veh = gt.boxes[np.isin(gt.cats, [4, 5, 6, 9])]
    with torch.inference_mode(), P.autocast(DEVICE):
        pred.set_image(im)
        # 1) кліки в центри машин (до 25 на кадр)
        for b in veh[rng.permutation(len(veh))[:25]]:
            c = np.array([[(b[0] + b[2]) / 2, (b[1] + b[3]) / 2]])
            m_, i_, _ = pred.predict(point_coords=c, point_labels=np.array([1]), multimask_output=True)
            k = int(np.argmax(i_))
            mbox = P.mask_to_box(m_[k] > 0)
            rows.append(dict(kind="машина", pred_iou=float(i_[k]), side_in=max(b[2] - b[0], b[3] - b[1]) * s,
                             box_iou=0.0 if mbox is None else float(P.box_iou(mbox, b)[0, 0]), box=b, path=p))
        # 2) кліки у фон: точка поза всіма GT-боксами й ignored regions (із запасом 10 px)
        allb = np.concatenate([gt.boxes, gt.ignore]) if len(gt.ignore) else gt.boxes
        n_bg = 0
        while n_bg < 15:
            x, y = rng.uniform(0, w), rng.uniform(0, h)
            if ((x > allb[:, 0] - 10) & (x < allb[:, 2] + 10) & (y > allb[:, 1] - 10) & (y < allb[:, 3] + 10)).any():
                continue
            m_, i_, _ = pred.predict(point_coords=np.array([[x, y]]), point_labels=np.array([1]), multimask_output=True)
            rows.append(dict(kind="фон", pred_iou=float(i_.max()), side_in=np.nan, box_iou=np.nan, box=None, path=p))
            n_bg += 1
st = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(7.5, 3.2))
for kind, col in [("машина", "#2bb5a0"), ("фон", "#e0703a")]:
    ax.hist(st[st.kind == kind].pred_iou, bins=np.linspace(0, 1, 41), alpha=.65, color=col, density=True,
            label=f"клік у {kind} (n={int((st.kind == kind).sum())})")
ax.set_xlabel("predicted_iou найкращої з 3 масок"); ax.set_ylabel("щільність"); ax.legend()
ax.set_title("predicted_iou оцінює якість маски, а не те, чи це об'єкт потрібного класу")
P.save_fig(fig, OUT / "01_pred_iou_bg_vs_obj.png")
for thr in (0.7, 0.8, 0.9):
    print(f"поріг predicted_iou ≥ {thr}: проходить {np.mean(st[st.kind == 'машина'].pred_iou >= thr):.0%} кліків у машини "
          f"і {np.mean(st[st.kind == 'фон'].pred_iou >= thr):.0%} кліків у фон")

In [ ]:
# Дрібні цілі: якість маски за розміром на вході SAM, повний кадр vs кроп навколо цілі.
# Кроп — 4 сторони цілі, але не менше 256 px: ціль займає ~1/4 входу енкодера, контекст навколо лишається.
obj = st[st.kind == "машина"].copy()
obj["bin"] = pd.cut(obj.side_in, [0, 8, 16, 32, np.inf], labels=["< 8", "8–16", "16–32", "≥ 32"])
crop_iou = []
for (p, grp) in tqdm(obj.groupby("path"), desc="кропи"):
    im = P.imread_rgb(p)
    h, w = im.shape[:2]
    for i, r in grp.iterrows():
        b = r.box
        CROP = int(max(256, 4 * max(b[2] - b[0], b[3] - b[1])))
        bx0 = int(np.clip((b[0] + b[2]) / 2 - CROP / 2, 0, max(0, w - CROP)))
        by0 = int(np.clip((b[1] + b[3]) / 2 - CROP / 2, 0, max(0, h - CROP)))
        c = im[by0:by0 + CROP, bx0:bx0 + CROP]
        with torch.inference_mode(), P.autocast(DEVICE):
            pred.set_image(c)                                       # кроп → 1024: ціль у кілька разів більша на вході енкодера
            pt = np.array([[(b[0] + b[2]) / 2 - bx0, (b[1] + b[3]) / 2 - by0]])
            m_, i_, _ = pred.predict(point_coords=pt, point_labels=np.array([1]), multimask_output=True)
        mbox = P.mask_to_box(m_[int(np.argmax(i_))] > 0)
        crop_iou.append((i, 0.0 if mbox is None else float(P.box_iou(mbox + [bx0, by0, bx0, by0], b)[0, 0])))
obj.loc[[i for i, _ in crop_iou], "box_iou_crop"] = [v for _, v in crop_iou]

tab = obj.groupby("bin", observed=True).agg(цілей=("box_iou", "size"), **{
    "IoU боксу маски, повний кадр": ("box_iou", "mean"), "IoU, кроп": ("box_iou_crop", "mean"),
    "частка IoU < 0.5, повний кадр": ("box_iou", lambda v: np.mean(v < 0.5)),
    "частка IoU < 0.5, кроп": ("box_iou_crop", lambda v: np.mean(v < 0.5))})
tab.index.name = f"сторона цілі на вході {SAM_INPUT}, px"
display(tab)
pred.set_image(img)                                                   # повертаємо основний кадр у кеш

**Підсумок розділу.**
* Клік у фон дає маску з `predicted_iou` не гіршим, ніж клік у машину: шматок дороги чи даху — теж валідний «об'єкт».
  `predicted_iou` — прогноз якості маски, а не впевненість у класі; поріг калібруйте на своїх даних (слайд 17).
* Чим менша ціль на вході енкодера, тим частіше маска вироджується в пляму або «розтікається» на сусідні об'єкти. Кроп, що
  збільшує ціль перед енкодером, повертає якість дрібним цілям без жодного донавчання — та сама ідея, що тайлінг для
  детектора (слайд 46). Великим цілям кроп не потрібен і навіть трохи шкодить: менше контексту навколо. Кропати варто
  за розміром цілі, а не все підряд.

## 6. Автоматичний режим: сітка точок замість промпта

`SAM2AutomaticMaskGenerator` ставить сітку `points_per_side²` точок, фільтрує маски за `pred_iou_thresh` і
`stability_score_thresh` і зводить NMS. Порівнюємо конфігурації зі слайду 23 і FastSAM (YOLO-seg «сегментуй усе»).
Кадр — з найбільшою кількістю дрібних машин (< 16 px на вході SAM). Метрики: **покриття машин** (бокс маски з IoU ≥ 0.5
до GT), окремо для дрібних, **частка «сміття»** (масок, bbox яких має IoU < 0.1 з кожним GT-боксом) і час.

In [ ]:
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator

CONFIGS = {
    "default (32², без кропів)": dict(points_per_side=32, crop_n_layers=0),
    "щільніша сітка 64²":         dict(points_per_side=64, crop_n_layers=0),
    "crop_n_layers=1":            dict(points_per_side=32, crop_n_layers=1, crop_n_points_downscale_factor=1),
    "кропи + min_area + м'якші пороги": dict(points_per_side=32, crop_n_layers=1, crop_n_points_downscale_factor=1,
                                             pred_iou_thresh=0.7, stability_score_thresh=0.9, min_mask_region_area=20),
}
# кадр із найбільшою кількістю дрібних машин: < 16 px на вході SAM
def n_small(it):
    p, gt = it
    v = gt.boxes[np.isin(gt.cats, [4, 5, 6, 9])]
    side = np.maximum(v[:, 2] - v[:, 0], v[:, 3] - v[:, 1]) * SAM_INPUT / max(P.image_size(p))
    return (side < 16).sum()

AUTO_PATH, GTA = max(items, key=n_small)
aimg = P.imread_rgb(AUTO_PATH)
veh = GTA.boxes[np.isin(GTA.cats, [4, 5, 6, 9])]
small_veh = np.maximum(veh[:, 2] - veh[:, 0], veh[:, 3] - veh[:, 1]) * SAM_INPUT / max(aimg.shape[:2]) < 16
print(f"{AUTO_PATH.name}: {aimg.shape[1]}×{aimg.shape[0]}, машин {len(veh)}, з них < 16 px на вході SAM: {small_veh.sum()}")
all_gt = GTA.boxes
auto_rows, auto_vis = [], {}


def score_masks(name, segs, ms):
    mb = np.array([P.mask_to_box(s_) for s_ in segs if s_.any()], np.float32).reshape(-1, 4)
    hit = (P.box_iou(veh, mb).max(1) >= 0.5) if len(mb) else np.zeros(len(veh), bool)
    junk = float((P.box_iou(mb, all_gt).max(1) < 0.1).mean()) if len(mb) else 0.0
    auto_rows.append({"режим": name, "масок": len(segs), "покриття машин": hit.mean(),
                      "покриття < 16 px": hit[small_veh].mean(), "сміття (не GT)": junk, "час, с": ms / 1000})
    auto_vis[name] = sorted(segs, key=lambda s_: -s_.sum())


for name, cfg in CONFIGS.items():
    gen = SAM2AutomaticMaskGenerator(pred.model, points_per_batch=128, **cfg)
    ts = []
    with torch.inference_mode(), P.autocast(DEVICE), P.timed(ts):
        try:
            anns = gen.generate(aimg)
        except IndexError:                  # sam2 падає, якщо жодна маска не пройшла пороги
            anns = []
    score_masks(name, [a["segmentation"] for a in anns], ts[0])
    del gen
    P.free_memory()

fs = P.load_ultra("fastsam")
fs.predict(aimg[..., ::-1].copy(), imgsz=1024, retina_masks=True, conf=0.4, iou=0.9, verbose=False, device=DEVICE)  # прогрів
ts = []
with P.timed(ts):
    r = fs.predict(aimg[..., ::-1].copy(), imgsz=1024, retina_masks=True, conf=0.4, iou=0.9, verbose=False, device=DEVICE)[0]
score_masks("FastSAM-s (YOLO-seg)", [] if r.masks is None else list(r.masks.data.cpu().numpy() > 0.5), ts[0])
del fs

auto_df = pd.DataFrame(auto_rows).set_index("режим")
display(auto_df.style.format({"покриття машин": "{:.0%}", "покриття < 16 px": "{:.0%}", "сміття (не GT)": "{:.0%}",
                              "час, с": "{:.2f}"}))
names = list(auto_vis)
P.show_row([P.overlay_masks(aimg, auto_vis[n][:400], alpha=.45, contour=False) for n in names],
           [f"{n}\n{auto_df.loc[n, 'масок']} масок · машин {auto_df.loc[n, 'покриття машин']:.0%} · "
            f"дрібних {auto_df.loc[n, 'покриття < 16 px']:.0%}" for n in names],
           size=4.6, ncols=3, save=OUT / "01_auto_modes.png")

**Підсумок розділу.**
* Без промпта SAM «не знає, що потрібно»: сотні масок, серед яких дороги, дахи, тіні й частини машин (слайд 18). Без
  фільтрації за площею, IoU і стабільністю це не розмітка, а шум.
* **Кропи** (`crop_n_layers`) піднімають покриття дрібних машин сильніше, ніж подвоєння сітки: кроп збільшує роздільність
  на вході енкодера, а щільніша сітка — ні (слайд 23). Ціна — кратно більший час і більше сміття. Навіть найкраща
  конфігурація покриває лише малу частку цілей < 16 px: для дрібних об'єктів потрібен промпт від детектора, а не сітка.
* FastSAM у рази швидший, але межі грубіші й дрібне губиться — «швидкість важливіша за межі» (слайд 19).

## 7. Де зникає час

Для кожної моделі окремо міряємо **енкодер** (раз на кадр) і **prompt encoder + декодер** на один бокс. Медіана після прогріву:
енкодер — 3 + 10 запусків, декодер — 30 вимірів (на CPU менше: 1 + 3 і 20). Те саме залізо, той самий кадр. Результат → `outputs/demo1/bench_sam.csv` (слайд 24).

In [ ]:
# основна модель розділів 2–6 більше не потрібна; pop — щоб клітинку можна було перезапускати
for _name in ("pred", "m"):
    globals().pop(_name, None)
P.free_memory()

bench_rows = []
for name in BENCH:
    enc_fn, dec_fn, info, model = P.sam_bench_fns(name, img, tgt, DEVICE)
    e, d = P.bench(enc_fn), P.bench(dec_fn, n_iter=30)
    bench_rows.append({"модель": P.SAM_ZOO[name]["label"], "енкодер, млн": info["enc_params"] / 1e6,
                       "енкодер, мс": e["median_ms"], "енкодер p90, мс": e["p90_ms"], "декодер, мс": d["median_ms"]})
    print(bench_rows[-1])
    del model, enc_fn, dec_fn
    P.free_memory()

# Декодер на CPU (як у браузерному демо SAM): той самий ViT-B, embedding уже порахований.
# Якщо весь ноутбук і так на CPU, цей рядок нічого не додає.
if DEVICE != "cpu":
    enc_fn, dec_fn, info, model = P.sam_bench_fns("sam-vit-base", img, tgt, "cpu")
    e, d = P.bench(enc_fn, n_warmup=1, n_iter=3), P.bench(dec_fn, n_iter=20)
    bench_rows.append({"модель": "SAM ViT-B · CPU", "енкодер, млн": info["enc_params"] / 1e6, "енкодер, мс": e["median_ms"],
                       "енкодер p90, мс": e["p90_ms"], "декодер, мс": d["median_ms"]})
    del model, enc_fn, dec_fn
    P.free_memory()

bench_df = pd.DataFrame(bench_rows)
bench_df["залізо"] = ["CPU" if m_.endswith("· CPU") else GPU for m_ in bench_df["модель"]]
bench_df.to_csv(OUT / "bench_sam.csv", index=False)
display(bench_df.round(1))

In [ ]:
gpu_df = bench_df[~bench_df["модель"].str.endswith("· CPU")]
fig, ax = plt.subplots(figsize=(10, 3.8))
x = np.arange(len(gpu_df))
ax.bar(x - .2, gpu_df["енкодер, мс"], .4, color="#2bb5a0", label="енкодер зображення")
ax.bar(x + .2, gpu_df["декодер, мс"], .4, color="#8a96a6", label="prompt encoder + декодер")
for i, r in gpu_df.reset_index().iterrows():
    ax.text(i - .2, r["енкодер, мс"], f"{r['енкодер, мс']:.0f}", ha="center", va="bottom", fontsize=8)
    ax.text(i + .2, r["декодер, мс"], f"{r['декодер, мс']:.1f}", ha="center", va="bottom", fontsize=8)
ax.axhline(1000 / 30, color="#e0703a", ls="--", lw=1.2)
ax.text(-.5, 1000 / 30 * 1.04, "33 мс = 30 FPS", color="#e0703a", ha="left", va="bottom", fontsize=9)
ax.set_ylim(0, gpu_df["енкодер, мс"].max() * 1.15)
ax.set_xticks(x, gpu_df["модель"], fontsize=8)
ax.set_ylabel("мс на кадр (медіана)")
ax.set_title(f"Де зникає час · {GPU} · кадр {W}×{H}")
ax.legend(loc="upper right")
P.save_fig(fig, OUT / "01_latency_budget.png")

### Амортизація: T(k) = T_enc + k · T_dec

Скільки коштує **одна маска**, якщо на кадр припадає k промптів (слайд 24). Для інтерактивної розмітки k великий — енкодер
розмивається; для «один промпт на кадр при 30 FPS» k = 1 і весь енкодер лягає на кожен кадр.

In [ ]:
k = np.arange(1, 201)
fig, ax = plt.subplots(figsize=(8, 3.6))
for _, r in bench_df.iterrows():
    T_k = r["енкодер, мс"] + k * r["декодер, мс"]           # T(k) = T_enc + k · T_dec
    ax.plot(k, T_k / k, label=r["модель"], ls="--" if "CPU" in r["модель"] else "-")
ax.axhline(1000 / 30, color="#e0703a", lw=1, ls=":")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("промптів на кадр, k"); ax.set_ylabel("мс на одну маску")
ax.set_title("Вартість маски при кешованому embedding")
ax.legend(fontsize=7, ncols=2)
P.save_fig(fig, OUT / "01_amortization.png")

r = bench_df.set_index("модель").loc[P.SAM_ZOO[BENCH[-1]]["label"]]
print(f"{BENCH[-1]}: 1 промпт на кадр = {r['енкодер, мс'] + r['декодер, мс']:.0f} мс; "
      f"100 промптів на кадр = {(r['енкодер, мс'] + 100 * r['декодер, мс']) / 100:.1f} мс на маску")

**Підсумок розділу.**
* Енкодер — майже весь час; декодер — одиниці мілісекунд на GPU і десятки на CPU. Декодер SAM 2.1 тут «важчий» за SAM 1,
  бо `predict()` ще й апсемплить маску до повного кадру й переводить її в numpy — у потоковому режимі цього немає. Тому SAM дешевий в інтерактивному
  інструменті (багато кліків на кадр) і дорогий у потоці (новий кадр — новий енкодер).
* SAM 2.1 T/S швидші за ViT-B при кращій якості — дефолт для нових пайплайнів; MobileSAM — у рази швидший енкодер за ту
  саму архітектуру декодера (дистиляція, заняття 15). Більший чекпойнт не лікує жодне з обмежень розділів 5–6 (слайд 18).
* Цифри зі статей (≈ 0.15 с ViT-H на A100) і ваші заміри — різні: міряйте на своєму залізі, з `synchronize()` і p90, а
  бортовий Jetson повільніший ще на порядок.

**Питання в аудиторію:** чому кропи в автоматичному режимі піднімають покриття сильніше, ніж подвоєння `points_per_side`,
і скільки разів зросте час для 4K-кадру з `crop_n_layers=1`?